In [ ]:
from appgeopy import *
from my_packages import *

# STAGE 1

## Export groundwater timeseries into feather file

## Assign English name to geospatial file

to help AI easily recognize the feather stem

## Export MLCW + GWL

I have asked AI to pair MLCW and closest GWL (with screen info)

now I need to export these into a specific folder for further usage

## Fill the missing timeseries at YIWU

## Replace new YIWU to `mlcw_gwl_timeseries`

## Extract leveling to get groundwater well elevation

## export groundwater well info to shapefile

then use it to extract the elevation from cokriging and taiwan dem 20m raster

## after elevation extraction, adjust the file, and export to machine-readable files

# STAGE 2

## Extract original MLCW grouped by layers

## Export MLCW + GWL

the latest file pairing MLCW and appropriate groundwater levels : `..\data\gwl\gwl_to_mlcw_layer_assignment_v3.csv`

# STAGE 3

## investigate and fill missing GWL data

## fill missing groundwater level values

In [ ]:
def hdf5_to_df(hdf5_data, select_station):
    date_arr = pd.to_datetime(
        [ele.decode("utf-8") for ele in hdf5_data["date"]]
    )
    cols = list(hdf5_data[select_station].keys())

    gwl_cache = {
        wellcode: hdf5_data[select_station][wellcode]["measure"]["model"]
        for wellcode in cols
    }
    df = pd.DataFrame(data=gwl_cache, index=date_arr)
    return df


# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -


def feather_to_df(fpath):
    df = pd.read_feather(fpath)
    df["datetime"] = pd.to_datetime(df["datetime"])
    df = df.set_index("datetime")
    return df


# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
def patch_column_gaps(df, column_mapping, inplace=True):
    """
    Fills missing values (NaN) in multiple target columns using their respective
    backup columns within the same DataFrame.

    Parameters:
    -----------
    df : pandas.DataFrame
        The DataFrame containing all columns.
    column_mapping : dict
        A dictionary where keys are target columns and values are backup columns.
        Format: {"target_col_1": "backup_col_1", "target_col_2": "backup_col_2"}
    inplace : bool, default True
        If True, modifies the DataFrame directly. If False, returns a modified copy.

    Returns:
    --------
    pandas.DataFrame or None
    """
    if not inplace:
        df = df.copy()

    # Loop through the dictionary and apply .fillna() to each pair automatically
    for target_col, backup_col in column_mapping.items():
        df[target_col] = df[target_col].fillna(df[backup_col])

    if not inplace:
        return df


# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
def patch_gaps_from_external_source(
    target_df, source_df, column_mapping, inplace=True
):
    """
    Fills missing values (NaN) in a target DataFrame using matching chronological
    values from columns in an external source DataFrame.

    Parameters:
    -----------
    target_df : pandas.DataFrame
        The primary DataFrame containing gaps that need to be filled.
    source_df : pandas.DataFrame
        The external source DataFrame containing the backup data columns.
    column_mapping : dict
        A dictionary mapping target column names to their external source column names.
        Format: {"target_well_code": "external_source_well_code"}
    inplace : bool, default True
        If True, modifies the target DataFrame directly. If False, returns a modified copy.

    Returns:
    --------
    pandas.DataFrame or None
    """
    if not inplace:
        target_df = target_df.copy()

    # Loop through the map and patch using the corresponding external column
    for target_col, source_col in column_mapping.items():
        # Pandas automatically aligns the values by their Datetime index matching rows
        target_df[target_col] = target_df[target_col].fillna(
            source_df[source_col]
        )

    if not inplace:
        return target_df

## open file

In [ ]:
hdf5_fpath = r"D:\1000_SCRIPTS\002_Project001\20240825_Model_GWL_fromHDF5\20240828_GWL_CRFP_model.h5"

h5_gwl_data, _ = gwatertools.open_HDF5(hdf5_fpath)

In [ ]:
topfolder = r"D:\1000_SCRIPTS\004_Project003\20260427_InSAR_MLCW_v2\data\gwl\well_timeseries"

select_station = "FANGCAO"
fpath = os.path.join(topfolder, f"{select_station}_gwl_timeseries.feather")


hdf5_df = hdf5_to_df(hdf5_data=h5_gwl_data, select_station=select_station)
feather_df = feather_to_df(fpath)

In [ ]:
feather_df.head(5)

In [ ]:
hdf5_df.head(5)

## visualize

In [ ]:
# feather_df.iloc[:, 0::2].plot()
feather_df.iloc[:, :].plot()

In [ ]:
hdf5_df.iloc[:, :].plot()

In [ ]:
feather_df.columns

## fill missing

### self filling

In [ ]:
# =================================================================
# SMART EXAMPLE USAGE
# =================================================================

# Define all your well dependencies clearly in one dictionary mapping
WELL_PATCH_MAP = {
    "09040212": "09040211",  # Target: 112, Backup: 111
    "09040222": "09040221",  # Target: 122, Backup: 121
    # "07230232": "07230231",  # Target: 122, Backup: 121
    # "07230242": "07230241",  # Target: 122, Backup: 121
}

# Process all updates simultaneously in a single line
final_updated_df = patch_column_gaps(feather_df, WELL_PATCH_MAP, inplace=False)

### cross filling

In [ ]:
final_updated_df.iloc[:, :].plot()

### save output

In [ ]:
final_updated_df.to_feather(fpath)